In [0]:
from typing import Dict
from pyspark.sql import functions as F
from dbx_vector_search import vector_topk
from dbx_keyword_search_bm25 import keyword_search  # <- change

INDEX_FQN = "tdis_dev_data_catalog.tdir.optimized_rag_chunks_vs"
CHUNKS_TABLE = "tdis_dev_data_catalog.tdir.optimized_rag_chunks"

def _clean_text_col(col):
    # light cleanup: whitespace + weird bullets + duplicated spaces
    return F.trim(
        F.regexp_replace(
            F.regexp_replace(col, r"\s+", " "),
            r"[•·]+", " "
        )
    )
    
def hybrid_search_rrf(
    spark,
    query: str,
    top_each: int = 10,
    top_n: int = 10,
    rrf_k: int = 60,
    return_with_text: bool = False,
):
    # Vector top_each
    vec_df = (vector_topk(spark, index_fqn=INDEX_FQN, query=query, top_k=top_each)
              .select("chunk_id", "search_score")
              .orderBy(F.col("search_score").desc())
              .limit(int(top_each)))
    vec_rank = [r["chunk_id"] for r in vec_df.select("chunk_id").collect()]

    # BM25 top_each
    kw_df = (keyword_search(spark, query=query, k=top_each, return_df=True)
             .select("chunk_id", "score")
             .orderBy(F.col("score").desc())
             .limit(int(top_each)))
    kw_rank = [r["chunk_id"] for r in kw_df.select("chunk_id").collect()]

    # RRF fuse + dedup
    scores: Dict[str, float] = {}
    src: Dict[str, set] = {}

    for i, cid in enumerate(vec_rank):
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (rrf_k + i + 1)
        src.setdefault(cid, set()).add("VEC")

    for i, cid in enumerate(kw_rank):
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (rrf_k + i + 1)
        src.setdefault(cid, set()).add("KW")

    fused = sorted(scores.items(), key=lambda x: -x[1])[:int(top_n)]
    out_rows = [(cid, float(s), "+".join(sorted(src.get(cid, set())))) for cid, s in fused]

    out_df = (spark.createDataFrame(out_rows, ["chunk_id", "rrf_score", "from"])
              .orderBy(F.col("rrf_score").desc()))

    if not return_with_text:
        return out_df

    chunks = spark.table(CHUNKS_TABLE).select("chunk_id", "text", "source_file", "chunk_index_in_file")
    return out_df.join(chunks, "chunk_id", "left").orderBy(F.col("rrf_score").desc())


In [0]:
query = "How did Harris County estimate the number of homes that flooded after Hurricane Harvey, and what data sources did they use?"

In [0]:
out = hybrid_search_rrf(spark, query, top_each=100, top_n=20, rrf_k=20, return_with_text=True)

In [0]:
display(out)

In [0]:
out_text_collection = out.orderBy(F.col("rrf_score").desc()).select(F.col("text")).limit(10).collect()

In [0]:
out_text_collection